# Phase 2: Data Normalization & Validation
Merge all datasets into unified master dataset with validation

In [23]:
import pandas as pd
import re
from typing import List
import os
import json
from pathlib import Path

## 1. Load All Datasets

**Critical:** Using EXACT column names verified in Phase 1

In [42]:
# Load all datasets
print("Loading datasets...\n")

ham_df = pd.read_csv('data/raw/spamassassin_ham_100.csv')
print(f"✅ SpamAssassin Ham: {len(ham_df)} emails")
print(f"   Columns: {', '.join(ham_df.columns)}")

phishbowl_df = pd.read_csv('data/raw/phishbowl_50.csv')
print(f"\n✅ Phishbowl: {len(phishbowl_df)} emails")
print(f"   Columns: {', '.join(phishbowl_df.columns)}")

# Prefer regenerated datasets if present (check both root and notebooks paths)
root = Path.cwd()
regen_candidates = [
    root / 'data' / 'regenerated_datasets',
    root / 'notebooks' / 'data' / 'regenerated_datasets'
]

def find_regen_path(filename: str) -> Path | None:
    for base in regen_candidates:
        candidate = base / filename
        if candidate.exists():
            return candidate
    return None

plain_json_path = find_regen_path('plain_llm_regenerated.json')
hybrid_json_path = find_regen_path('hybrid_vtriad_regenerated.json')

if plain_json_path:
    with open(plain_json_path, 'r') as f:
        plain_llm_data = json.load(f)
    plain_llm_df = pd.DataFrame(plain_llm_data)
    print(f"\n✅ Plain LLM (Regenerated JSON): {len(plain_llm_df)} emails")
    print(f"   Path: {plain_json_path}")
    print(f"   Columns: {', '.join(plain_llm_df.columns)}")
else:
    plain_llm_df = pd.read_csv('data/raw/plain_llm_phishing.csv')
    print(f"\n✅ Plain LLM (Legacy CSV): {len(plain_llm_df)} emails")
    print(f"   Columns: {', '.join(plain_llm_df.columns)}")

if hybrid_json_path:
    with open(hybrid_json_path, 'r') as f:
        hybrid_data = json.load(f)
    hybrid_df = pd.DataFrame(hybrid_data)
    print(f"\n✅ Hybrid V-Triad (Regenerated JSON): {len(hybrid_df)} emails")
    print(f"   Path: {hybrid_json_path}")
    print(f"   Columns: {', '.join(hybrid_df.columns)}")
else:
    hybrid_df = pd.read_csv('data/raw/hybrid_vtriad_phishing.csv')
    print(f"\n✅ Hybrid V-Triad (Legacy CSV): {len(hybrid_df)} emails")
    print(f"   Columns: {', '.join(hybrid_df.columns)}")

Loading datasets...

✅ SpamAssassin Ham: 100 emails
   Columns: id, subject, sender, body

✅ Phishbowl: 50 emails
   Columns: id, subject, sender, body

✅ Plain LLM (Regenerated JSON): 50 emails
   Path: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\notebooks\data\regenerated_datasets\plain_llm_regenerated.json
   Columns: subject, sender, body, urls

✅ Hybrid V-Triad (Regenerated JSON): 50 emails
   Path: c:\Users\datta\Documents\SEM6\Capstone\Preprocessing2\notebooks\data\regenerated_datasets\hybrid_vtriad_regenerated.json
   Columns: subject, sender, body, urls


## 2. Validate Content BEFORE Merging

**Lesson from Phase 1:** Always validate IMMEDIATELY after loading

In [26]:
def validate_dataset(df: pd.DataFrame, name: str) -> bool:
    """Validate dataset has non-empty content."""
    print(f"\n{'='*60}")
    print(f"Validating: {name}")
    print(f"{'='*60}")
    
    # Check required columns
    required_cols = ['subject', 'sender', 'body']
    missing_cols = [col for col in required_cols if col not in df.columns]
    
    if missing_cols:
        print(f"❌ FAIL: Missing columns: {missing_cols}")
        return False
    
    print(f"✅ Required columns present")
    
    # Check for nulls
    null_counts = df[required_cols].isnull().sum()
    if null_counts.sum() > 0:
        print(f"❌ FAIL: Null values detected:")
        print(null_counts[null_counts > 0])
        return False
    
    print(f"✅ No null values")
    
    # Check body content length
    body_lengths = df['body'].astype(str).str.len()
    empty_bodies = (body_lengths < 50).sum()
    
    print(f"\n📏 Body Length Stats:")
    print(f"   Mean: {body_lengths.mean():.1f} chars")
    print(f"   Min: {body_lengths.min()} chars")
    print(f"   Max: {body_lengths.max()} chars")
    
    if empty_bodies > 0:
        print(f"\n⚠️  WARNING: {empty_bodies} emails have very short bodies (<50 chars)")
        # Show samples of short bodies
        short_bodies = df[body_lengths < 50]
        print(f"\nSample short bodies:")
        for idx, row in short_bodies.head(3).iterrows():
            print(f"  ID {row.get('id', idx)}: '{row['body'][:100]}'")
    else:
        print(f"✅ All bodies have substantial content (≥50 chars)")
    
    # Sample content
    print(f"\n📧 Sample Email:")
    sample = df.iloc[0]
    print(f"   Subject: {sample['subject']}")
    print(f"   Sender: {sample['sender']}")
    print(f"   Body (first 150 chars): {sample['body'][:150]}...")
    
    print(f"\n✅ VALIDATION PASSED: {name}")
    return True

In [27]:
# Validate all datasets
datasets = [
    (ham_df, 'SpamAssassin Ham'),
    (phishbowl_df, 'Phishbowl'),
    (plain_llm_df, 'Plain LLM'),
    (hybrid_df, 'Hybrid V-Triad')
]

all_valid = True
for df, name in datasets:
    if not validate_dataset(df, name):
        all_valid = False

if all_valid:
    print(f"\n\n{'='*60}")
    print("✅ ALL DATASETS VALIDATED - PROCEEDING TO MERGE")
    print(f"{'='*60}")
else:
    print(f"\n\n{'='*60}")
    print("❌ VALIDATION FAILED - FIX ISSUES BEFORE PROCEEDING")
    print(f"{'='*60}")
    raise ValueError("Dataset validation failed")


Validating: SpamAssassin Ham
✅ Required columns present
✅ No null values

📏 Body Length Stats:
   Mean: 1753.2 chars
   Min: 60 chars
   Max: 16464 chars
✅ All bodies have substantial content (≥50 chars)

📧 Sample Email:
   Subject: Re: New gkrellm 2.0.0, gtk2 version
   Sender: Brian Fahrlander <kilroy@kamakiriad.com>
   Body (first 150 chars): On Mon, 26 Aug 2002 19:14:54 +0200, Matthias Saou <matthias@egwn.net> wrote:

> Hi all,
> 
> I've repackaged the new gkrellm 2.0.0 which is now ported...

✅ VALIDATION PASSED: SpamAssassin Ham

Validating: Phishbowl
✅ Required columns present
✅ No null values

📏 Body Length Stats:
   Mean: 967.6 chars
   Min: 17 chars
   Max: 4598 chars

⚠️  WARNING: 5 emails have very short bodies (<50 chars)

Sample short bodies:
  ID 7: 'Message Body [Blank] Attachment'
  ID 23: '[No body content]'
  ID 29: '[No body content]'

📧 Sample Email:
   Subject: **Emergency: Help Needed for COVID-19 Variant Contact Tracing**
   Sender: [Unknown]
   Body (first 150

## 3. Add Metadata Columns

Add `source` and `actual_class` for traceability

In [28]:
# Add source and class labels
ham_df['source'] = 'spamassassin_ham'
ham_df['actual_class'] = 0  # Benign

phishbowl_df['source'] = 'phishbowl'
phishbowl_df['actual_class'] = 1  # Phishing

plain_llm_df['source'] = 'plain_llm'
plain_llm_df['actual_class'] = 1  # Phishing

hybrid_df['source'] = 'hybrid_vtriad'
hybrid_df['actual_class'] = 1  # Phishing

print("✅ Added metadata columns: 'source' and 'actual_class'")

✅ Added metadata columns: 'source' and 'actual_class'


## 4. Extract URLs from Body Content

Extract all URLs for future cue detection

In [43]:
def extract_urls(text: str) -> List[str]:
    """Extract all URLs from text."""
    # Match http/https URLs
    url_pattern = r'https?://[^\s<>"\']+'  
    urls = re.findall(url_pattern, str(text))
    return urls if urls else []

# Apply to all datasets
print("Extracting URLs...")

ham_df['extracted_urls'] = ham_df['body'].apply(extract_urls)
phishbowl_df['extracted_urls'] = phishbowl_df['body'].apply(extract_urls)
plain_llm_df['extracted_urls'] = plain_llm_df['body'].apply(extract_urls)
hybrid_df['extracted_urls'] = hybrid_df['body'].apply(extract_urls)

print("\n📊 URL Extraction Summary:")
print(f"  Ham: {ham_df['extracted_urls'].apply(len).sum()} URLs")
print(f"  Phishbowl: {phishbowl_df['extracted_urls'].apply(len).sum()} URLs")
print(f"  Plain LLM: {plain_llm_df['extracted_urls'].apply(len).sum()} URLs")
print(f"  Hybrid: {hybrid_df['extracted_urls'].apply(len).sum()} URLs")
print(f"\n✅ URL extraction complete")

Extracting URLs...

📊 URL Extraction Summary:
  Ham: 179 URLs
  Phishbowl: 18 URLs
  Plain LLM: 0 URLs
  Hybrid: 0 URLs

✅ URL extraction complete


## 5. Standardize Schema & Merge

Keep only common columns across all datasets

In [30]:
# Define standard column order
standard_cols = ['subject', 'sender', 'body', 'extracted_urls', 'source', 'actual_class']

# Select only standard columns (drop dataset-specific columns)
ham_clean = ham_df[standard_cols].copy()
phishbowl_clean = phishbowl_df[standard_cols].copy()
plain_llm_clean = plain_llm_df[standard_cols].copy()
hybrid_clean = hybrid_df[standard_cols].copy()

print(f"✅ Standardized schema to: {', '.join(standard_cols)}")

✅ Standardized schema to: subject, sender, body, extracted_urls, source, actual_class


In [44]:
# Concatenate all datasets
master_df = pd.concat([
    ham_clean,
    phishbowl_clean,
    plain_llm_clean,
    hybrid_clean
], ignore_index=True)

# Add global email_id
master_df.insert(0, 'email_id', range(1, len(master_df) + 1))

print(f"\n✅ Merged all datasets into master_df")
print(f"\n📊 Master Dataset Summary:")
print(f"   Total emails: {len(master_df)}")
print(f"   Columns: {', '.join(master_df.columns)}")
print(f"\n📈 Class Distribution:")
print(master_df['actual_class'].value_counts().sort_index())
print(f"\n📂 Source Distribution:")
print(master_df['source'].value_counts())


✅ Merged all datasets into master_df

📊 Master Dataset Summary:
   Total emails: 250
   Columns: email_id, subject, sender, body, extracted_urls, source, actual_class

📈 Class Distribution:
actual_class
0    100
1    150
Name: count, dtype: int64

📂 Source Distribution:
source
spamassassin_ham    100
phishbowl            50
plain_llm            50
hybrid_vtriad        50
Name: count, dtype: int64


## 6. Final Validation

**Critical:** Validate merged dataset before saving

In [32]:
print("\n" + "="*80)
print("FINAL VALIDATION - MASTER DATASET")
print("="*80 + "\n")

# Check expected counts
expected_total = 250  # 100 ham + 150 phishing
expected_benign = 100
expected_phishing = 150

actual_total = len(master_df)
actual_benign = (master_df['actual_class'] == 0).sum()
actual_phishing = (master_df['actual_class'] == 1).sum()

print(f"📊 Row Count Validation:")
print(f"   Expected total: {expected_total}")
print(f"   Actual total: {actual_total}")
assert actual_total == expected_total, f"❌ Row count mismatch!"
print(f"   ✅ PASS\n")

print(f"📊 Class Distribution Validation:")
print(f"   Expected benign: {expected_benign}")
print(f"   Actual benign: {actual_benign}")
assert actual_benign == expected_benign, f"❌ Benign count mismatch!"
print(f"   ✅ PASS")

print(f"\n   Expected phishing: {expected_phishing}")
print(f"   Actual phishing: {actual_phishing}")
assert actual_phishing == expected_phishing, f"❌ Phishing count mismatch!"
print(f"   ✅ PASS\n")

# Check for nulls
null_counts = master_df.isnull().sum()
if null_counts.sum() > 0:
    print(f"⚠️  Null values detected:")
    print(null_counts[null_counts > 0])
else:
    print(f"✅ No null values detected\n")

# Check body content
body_lengths = master_df['body'].astype(str).str.len()
short_bodies = (body_lengths < 50).sum()

print(f"📏 Content Length Validation:")
print(f"   Mean body length: {body_lengths.mean():.1f} chars")
print(f"   Emails with short bodies (<50 chars): {short_bodies}")

if short_bodies > 0:
    print(f"   ⚠️  WARNING: Some emails have short bodies")
else:
    print(f"   ✅ PASS\n")

# Sample from each source
print(f"\n📧 Sample Emails (one from each source):\n")
for source in master_df['source'].unique():
    sample = master_df[master_df['source'] == source].iloc[0]
    print(f"  {source}:")
    print(f"    Subject: {sample['subject'][:50]}...")
    print(f"    Body: {sample['body'][:100]}...")
    print(f"    Class: {sample['actual_class']} | URLs: {len(sample['extracted_urls'])}\n")

print("="*80)
print("✅ VALIDATION COMPLETE - MASTER DATASET IS READY")
print("="*80)


FINAL VALIDATION - MASTER DATASET

📊 Row Count Validation:
   Expected total: 250
   Actual total: 250
   ✅ PASS

📊 Class Distribution Validation:
   Expected benign: 100
   Actual benign: 100
   ✅ PASS

   Expected phishing: 150
   Actual phishing: 150
   ✅ PASS

✅ No null values detected

📏 Content Length Validation:
   Mean body length: 1256.1 chars
   Emails with short bodies (<50 chars): 5
   ⚠️  WARNING: Some emails have short bodies

📧 Sample Emails (one from each source):

  spamassassin_ham:
    Subject: Re: New gkrellm 2.0.0, gtk2 version...
    Body: On Mon, 26 Aug 2002 19:14:54 +0200, Matthias Saou <matthias@egwn.net> wrote:

> Hi all,
> 
> I've re...
    Class: 0 | URLs: 4

  phishbowl:
    Subject: **Emergency: Help Needed for COVID-19 Variant Cont...
    Body: This phish typically originates from a compromised Cornell account that has since been secured. The ...
    Class: 1 | URLs: 1

  plain_llm:
    Subject: Action Required: Security Policy Update...
    Body: Dear T

## 7. Export Master Dataset

In [45]:
# Create processed directory if it doesn't exist
os.makedirs('data/processed', exist_ok=True)

# Export to CSV
output_path = 'data/processed/master_emails.csv'
master_df.to_csv(output_path, index=False)

print(f"\n✅ Master dataset saved to: {output_path}")
print(f"\n📊 Final Stats:")
print(f"   Total emails: {len(master_df)}")
print(f"   Benign: {actual_benign}")
print(f"   Phishing: {actual_phishing}")
print(f"   Columns: {len(master_df.columns)}")
print(f"   File size: {os.path.getsize(output_path) / 1024:.1f} KB")


✅ Master dataset saved to: data/processed/master_emails.csv

📊 Final Stats:
   Total emails: 250
   Benign: 100
   Phishing: 150
   Columns: 7
   File size: 343.2 KB


## Summary

In [22]:
print("\n" + "="*80)
print("PHASE 2 COMPLETE: DATA NORMALIZATION & VALIDATION")
print("="*80 + "\n")
print("✅ All 4 datasets loaded and validated")
print("✅ URLs extracted from all emails")
print("✅ Metadata columns added (source, actual_class)")
print("✅ Schema standardized across all sources")
print("✅ Master dataset created and validated")
print("✅ Exported to data/processed/master_emails.csv")
print("\n📌 Next Phase: Cue Pattern Development (Phase 3)")
print("="*80)


PHASE 2 COMPLETE: DATA NORMALIZATION & VALIDATION

✅ All 4 datasets loaded and validated
✅ URLs extracted from all emails
✅ Metadata columns added (source, actual_class)
✅ Schema standardized across all sources
✅ Master dataset created and validated
✅ Exported to data/processed/master_emails.csv

📌 Next Phase: Cue Pattern Development (Phase 3)
